In [0]:
CREATE SCHEMA IF NOT EXISTS orderdb;


In [0]:
DROP TABLE IF EXISTS orderdb.shipments ;


In [0]:
-- starts with 4 columns, say an year ago
CREATE OR REPLACE TABLE orderdb.shipments (
  shipment_id INT,
  product_id STRING,
  quantity INT,
  country STRING
);

In [0]:
DESCRIBE orderdb.shipments;

In [0]:
-- right data, data, data types validated

INSERT INTO orderdb.shipments VALUES
(1001,'P101',5,'India'),
(1002,'P102',8,'USA');

In [0]:
-- Schema Enforcement, in delta tables, it ensure that all parquet adhere to schema
-- add one extra column which does not exist, this cause error
-- error expected here, schema mismatch
INSERT INTO orderdb.shipments VALUES
(1003,'P103',10,'UK','Air');

com.databricks.sql.transaction.tahoe.DeltaAnalysisException: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 813c54b2-ab6a-40e8-8368-389f653af144).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- shipment_id: integer (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- country: string (nullable = true)


Data schema:
root
-- shipment_id: integer (nullable = true)
-- product_id: string (nullable = true)
-- quantity: integer (nullable = true)
-- country: string (nullable = true)
-- col5: string (nullable = true)

         
Table ACLs are enabled in this cluster, so automatic schema migration is not allowed. Please use the ALTER TA

In [0]:
-- wrong data type
-- error expected
INSERT INTO orderdb.shipments VALUES
(1004,'P104','abc','Germany');

org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 2, position 1) ==
INSERT INTO orderdb.shipments VALUES
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:188)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:51)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2(Cast.scala:973)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2$adapted(Cast.scala:973)
	at org.apache.spark.sql.catalyst.expressions.Cast.buildCast(Cast.scala:681)
	at org.apache.spark.sql.catalyst

In [0]:
-- Wrong column order danger
-- error expected
INSERT INTO orderdb.shipments
SELECT 'Japan',15,'P105',1005;

org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'Japan' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== SQL (line 3, position 1) ==
INSERT INTO orderdb.shipments
^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:188)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:51)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2(Cast.scala:973)
	at org.apache.spark.sql.catalyst.expressions.Cast.$anonfun$castToInt$2$adapted(Cast.scala:973)
	at org.apache.spark.sql.catalyst.expressions.Cast.buildCast(Cast.scala:681)
	at org.apache.spark.sql.catalyst.expr

In [0]:
-- Schema Evolution — Add New Column
-- say, 6 months ago

ALTER TABLE orderdb.shipments
ADD COLUMNS (transport_mode STRING);

In [0]:
-- post schema evolution, added new columns, 
-- remember old data files do not have this new column, so it will be null
INSERT INTO orderdb.shipments VALUES
(1006,'P106',20,'France','Air');

In [0]:
SELECT * FROM orderdb.shipments;

In [0]:
-- auto evoluation with merge, means automatic column creation, like you don't do alter table yourself
-- we are adding shipped date

CREATE OR REPLACE TEMP VIEW new_shipments AS
SELECT 1007 shipment_id,
       'P107' product_id,
       30 quantity,
       'Canada' country,
       'Sea' transport_mode,
       current_date() shipped_date;

In [0]:
select * from new_shipments

In [0]:
SET spark.databricks.delta.schema.autoMerge.enabled=true;
-- important, else will not work

In [0]:
DESCRIBE orderdb.shipments;
-- before marge, we don't see shipment date

In [0]:
MERGE INTO orderdb.shipments t
USING new_shipments s
ON t.shipment_id=s.shipment_id
WHEN NOT MATCHED THEN
INSERT *

-- will create shipment_date column automatically

In [0]:
DESCRIBE orderdb.shipments;
-- post merge, we   see shipment date

In [0]:
SELECT * FROM orderdb.shipments

In [0]:
%python
# data frame example

from pyspark.sql.types import *

schema = StructType([
 StructField("shipment_id", IntegerType(), True),
 StructField("product_id", StringType(), True),
 StructField("quantity", IntegerType(), True),
 StructField("country", StringType(), True),
 StructField("transport_mode", StringType(), True),
 StructField("priority", StringType(), True)
])

# note, we are adding priority column

data=[
(1008,'P108',50,'Japan','Road','High')
]

df = spark.createDataFrame(data, schema)

df.write \
 .format("delta") \
 .mode("append") \
 .option("mergeSchema","true") \
 .saveAsTable("orderdb.shipments")

In [0]:
DESCRIBE orderdb.shipments;

In [0]:
ALTER TABLE orderdb.shipments SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
-- for rename

In [0]:
-- rename column

ALTER TABLE orderdb.shipments
RENAME COLUMN country TO destination_country;


In [0]:
DESCRIBE orderdb.shipments;

In [0]:
-- drop column

ALTER TABLE orderdb.shipments
DROP COLUMN priority;

In [0]:
DESCRIBE orderdb.shipments;

In [0]:
-- Enforce Constraints  
ALTER TABLE orderdb.shipments
ADD CONSTRAINT qty_positive
CHECK (quantity > 0);

In [0]:
INSERT INTO orderdb.shipments
VALUES
(1010,'P110',-5,'India','Air', current_date());

-- see, we try to insert -5 for quantity
-- should fail

com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException: [DELTA_VIOLATE_CONSTRAINT_WITH_VALUES] CHECK constraint qty_positive (quantity > 0) violated by row with values:
 - quantity : -5
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.getConstraintViolationWithValuesException(InvariantViolationException.scala:82)
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:110)
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException$.apply(InvariantViolationException.scala:121)
	at com.databricks.sql.transaction.tahoe.schema.DeltaInvariantViolationException.apply(InvariantViolationException.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at com.databricks.sql.transaction.tahoe.constraints.DeltaInvariantCheckerExec.$anonfun$doExecute$4(DeltaInvariantCheckerExec.scala:118)
	at scala

In [0]:
INSERT INTO orderdb.shipments
VALUES
(2020,'P220',20,'India','Ship', current_date());
-- should pass constrain

-- see, we try to insert -5 for quantity
-- should fail

In [0]:
SHOW TBLPROPERTIES orderdb.shipments;

In [0]:
DESCRIBE DETAIL orderdb.shipments;

In [0]:
ALTER TABLE orderdb.shipments
DROP CONSTRAINT qty_positive;

--remove constraints